# 03 - Train the SciBERT relation classifier (typed entity markers)

Each ordered pair of entities in a sentence is encoded with typed markers such as
`<S:Method> ... </S:Method>` and `<O:Task> ... </O:Task>`; the two opening-marker states are
concatenated and classified into the seven relation types or `NO-RELATION`.

* Experiment A: candidate pairs from gold entities (relation classification only).
* Experiment B: candidate pairs from the NER predictions of notebook 02 (end to end).

Run notebook 02 first, so that `OUTPUTS/ner/predictions_{dev,test}.jsonl` exist.

In [ ]:
# Environment setup: works on Kaggle, Google Colab and a local checkout.
import os, subprocess, sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HaoudarNouhaila/scigraph-nlp.git"

if IN_KAGGLE or IN_COLAB:
    base = Path("/kaggle/working") if IN_KAGGLE else Path("/content")
    ROOT = base / "scigraph-nlp"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("project root:", ROOT.name)

In [ ]:
# Download the official SciERC release (once) and build sentence-level splits.
if not Path("data/processed/train.jsonl").exists():
    if not Path("data/raw/scierc/train.json").exists():
        subprocess.run([sys.executable, "scripts/download_scierc.py", "--out", "data/raw/scierc"], check=True)
    subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)
print(sorted(p.name for p in Path("data/processed").glob("*.jsonl")))

In [ ]:
# Persistent output directory so that an interrupted session can resume training.
# Colab: Google Drive. Kaggle: attach the output of a previous run of this notebook as an
# input dataset and it is copied back before training starts.
import shutil
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUTS = Path("/content/drive/MyDrive/scigraph-nlp/outputs")
else:
    OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)
if IN_KAGGLE:
    for prev in Path("/kaggle/input").glob("*/outputs"):
        print("restoring checkpoints from", prev)
        shutil.copytree(prev, OUTPUTS, dirs_exist_ok=True)
import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
CONFIG = "configs/re.yaml" if torch.cuda.is_available() else "configs/re_cpu.yaml"
cmd = [sys.executable, "-m", "scigraph.training.train_re", "--config", CONFIG,
       "--set", f"output_dir={OUTPUTS / 're'}", f"ner_predictions_dir={OUTPUTS / 'ner'}"]
_ = subprocess.run(cmd, check=True)

In [ ]:
import json, pandas as pd
m = json.loads((OUTPUTS / "re" / "metrics.json").read_text())
rows = []
a = m["experiment_a_gold_entities"]["test"]
rows.append(("A: gold entities", "boundaries", *[a["boundaries"]["micro"][k] for k in ("precision", "recall", "f1")]))
b = m["experiment_b_predicted_entities"]
if isinstance(b, dict):
    for crit in ("boundaries", "strict"):
        rows.append(("B: predicted entities", crit, *[b["test"][crit]["micro"][k] for k in ("precision", "recall", "f1")]))
pd.DataFrame(rows, columns=["setting", "criterion", "P", "R", "F1"])